In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("transactions.csv", sep=";")

In [ ]:
cancel_df = df[df["status"] == "cancelled"]

cancel_stage_summary = (
    cancel_df["cancel_stage"]
    .value_counts()
    .to_frame(name="total_cancelled")
)

cancel_stage_summary["percentage"] = (
    cancel_stage_summary["total_cancelled"] /
    cancel_stage_summary["total_cancelled"].sum()
) * 100

cancel_stage_summary["percentage"] = cancel_stage_summary["percentage"].round(2)

cancel_stage_summary

,total_cancelled,percentage
cancel_stage,,
payment,3025,69.96
detail,847,19.59
browse,452,10.45


In [ ]:
total_batal = cancel_df.shape[0]
print(f"Jumlah keseluruhan pesanan yang batal {total_batal}")

Jumlah keseluruhan pesanan yang batal 4324


In [ ]:
payment_cancel_df = df.loc[
    (df["status"] == "cancelled") &
    (df["cancel_stage"] == "payment")
]

payment_cancel_summary = (
    payment_cancel_df
        .groupby("seabank_user")
        .size()
        .to_frame(name="total_payment_cancellations")
        .reset_index()
)

total_payment_cancel = payment_cancel_summary["total_payment_cancellations"].sum()

payment_cancel_summary["percentage"] = (
    payment_cancel_summary["total_payment_cancellations"] / total_payment_cancel
) * 100

payment_cancel_summary["percentage"] = payment_cancel_summary["percentage"].round(2)

# Tambahkan baris ini untuk mengganti nilai
payment_cancel_summary["seabank_user"] = payment_cancel_summary["seabank_user"].replace({False: "Tidak", True: "Ya"})

payment_cancel_summary

,seabank_user,total_payment_cancellations,percentage
0,Tidak,2191,72.43
1,Ya,834,27.57


In [ ]:
df["is_cancelled"] = df["status"].eq("cancelled")

segment_analysis = (
    df
    .groupby(["age_group", "seabank_user"], as_index=False)
    .agg(
        total_transactions=("status", "count"),
        total_cancelled=("is_cancelled", "sum"),
    )
)

segment_analysis["cancel_rate"] = (
    segment_analysis["total_cancelled"] /
    segment_analysis["total_transactions"] * 100
).round(2)

segment_analysis = segment_analysis.sort_values(
    by="total_cancelled",
    ascending=False
)

segment_analysis["seabank_user"] = segment_analysis["seabank_user"].replace({False: "Tidak", True: "Ya"})

segment_analysis

,age_group,seabank_user,total_transactions,total_cancelled,cancel_rate
2,25-34,Tidak,4934,1186,24.04
0,18-24,Tidak,3887,916,23.57
4,35-44,Tidak,2615,646,24.70
3,25-34,Ya,2967,524,17.66
1,18-24,Ya,2092,349,16.68
6,45+,Tidak,1367,348,25.46
5,35-44,Ya,1494,249,16.67
7,45+,Ya,644,106,16.46


In [ ]:
def categorize_session(hour):
    if 11 <= hour <= 13:
        return "Makan Siang (11.00-13.00)"
    if 18 <= hour <= 20:
        return "Makan Malam (18.00-20.00)"
    if 0 <= hour <= 5:
        return "Dini Hari (00.00-06.00)"
    return "Lain-lain"

df["session"] = df["transaction_hour"].apply(categorize_session)

payment_cancel_df = df.loc[
    (df["status"] == "cancelled") &
    (df["cancel_stage"] == "payment")
]

session_summary = (
    payment_cancel_df
        .groupby("session", as_index=False)
        .size()
        .rename(columns={"size": "total_payment_cancellations"})
)

total_payment_cancel = session_summary["total_payment_cancellations"].sum()

session_summary["percentage"] = (
    session_summary["total_payment_cancellations"] / total_payment_cancel * 100
).round(2)

peak_hour_summary = (
    payment_cancel_df
        .groupby(["session", "transaction_hour"], as_index=False)
        .size()
        .rename(columns={"size": "payment_cancellations"})
        .sort_values(["session", "payment_cancellations"], ascending=[True, False])
        .drop_duplicates("session")
        [["session", "transaction_hour"]]
)

final_session_analysis = (
    session_summary
        .merge(peak_hour_summary, on="session", how="left")
        .sort_values(by="total_payment_cancellations", ascending=False)
)

final_session_analysis

,session,total_payment_cancellations,percentage,transaction_hour
1,Lain-lain,1351,44.66,16
0,Dini Hari (00.00-06.00),707,23.37,5
3,Makan Siang (11.00-13.00),496,16.40,12
2,Makan Malam (18.00-20.00),471,15.57,19


In [ ]:
payment_cancel_df = df.loc[
    (df["status"] == "cancelled") &
    (df["cancel_stage"] == "payment")
]

hourly_analysis = (
    payment_cancel_df
        .groupby("transaction_hour", as_index=False)
        .size()
        .rename(columns={"size": "payment_cancellations"})
        .sort_values(by="payment_cancellations", ascending=False)
)

total_payment_cancel = hourly_analysis["payment_cancellations"].sum()

hourly_analysis["percentage"] = (
    hourly_analysis["payment_cancellations"] / total_payment_cancel * 100
).round(2)

top_n = 5
top_hours = hourly_analysis.head(top_n)["transaction_hour"].tolist()

hourly_analysis["peak_flag"] = hourly_analysis["transaction_hour"].isin(top_hours)

hourly_analysis.head(10)

,transaction_hour,payment_cancellations,percentage,peak_flag
19,19,201,6.64,True
12,12,198,6.55,True
13,13,178,5.88,True
18,18,166,5.49,True
16,16,141,4.66,True
5,5,130,4.30,False
8,8,126,4.17,False
3,3,125,4.13,False
2,2,125,4.13,False
21,21,121,4.00,False


In [ ]:
top_hours = hourly_analysis.head(5)["transaction_hour"].tolist()

top_hours

[19, 12, 13, 18, 16]

In [ ]:
payment_cancel_df = df.loc[
    (df["status"] == "cancelled") &
    (df["cancel_stage"] == "payment")
]

payment_cancel_city = (
    payment_cancel_df
        .groupby("city", as_index=False)
        .size()
        .rename(columns={"size": "total_payment_cancellations"})
        .sort_values(by="total_payment_cancellations", ascending=False)
)

total_payment_cancel = payment_cancel_city["total_payment_cancellations"].sum()

payment_cancel_city["percentage"] = (
    payment_cancel_city["total_payment_cancellations"] / total_payment_cancel * 100
).round(2)

payment_cancel_city

,city,total_payment_cancellations,percentage
1,Jakarta,1091,36.07
3,Medan,600,19.83
4,Surabaya,598,19.77
0,Bandung,417,13.79
2,Makassar,319,10.55
